In [33]:
import numpy as np
import torch
import torch.nn as nn

Comprobar que detecte GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


## Embedding Posicional

In [35]:
class EmbeddingPosicional(nn.Module):
    def __init__(self, d_embedding, vocab_size, seq_len):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(1, seq_len, d_embedding) * 0.02) # Inicialización pequeña
        self.embedding = nn.Embedding(vocab_size, d_embedding)
    
    def forward(self, x):
        out = self.embedding(x) # devuelve (b, n, d)
        _, n, _ = out.shape
        out = out + self.weights[:,:n,:]
        return out

## Capa FeedForward

In [36]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.l1 = nn.Linear(d_model, d_model*4)
        self.silu = nn.SiLU()
        self.l2 = nn.Linear(d_model*4, d_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm(x)
        x = self.l1(x)
        x = self.silu(x)
        x = self.l2(x)
        x = self.dropout(x)
        return x

## Capa de atención

In [37]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, n_head, d_head, use_mask=False):
        super().__init__()
        self.qw = nn.Linear(d_model, d_head*n_head)
        self.kw = nn.Linear(d_model, d_head*n_head)
        self.vw = nn.Linear(d_model, d_head*n_head)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)
        self.out = nn.Linear(d_head*n_head, d_model)

        self.n_head = n_head
        self.d_head = d_head
        self.scale = d_head ** 0.5
        self.use_mask = use_mask
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        b, n, d = x.shape
        x = self.norm(x)

        q = self.qw(x)
        k = self.kw(x)
        v = self.vw(x)

        q = q.reshape(b, -1, self.n_head, self.d_head)
        k = k.reshape(b, -1, self.n_head, self.d_head)
        v = v.reshape(b, -1, self.n_head, self.d_head)

        #print(q.shape)
        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale
        
        if self.use_mask:
            _, _, _, n = scores.shape
            mask = torch.tril(torch.ones(1, 1, n, n))   # (n, n)

            scores = scores.masked_fill(mask[:, :, :n, :n] == 0, float('-inf'))

        softmax_scores = scores.softmax(dim=-1)
        att = self.dropout(softmax_scores)
        out = torch.einsum('bhij,bjhd->bihd', att, v)
        #print(out.shape)
        out = self.dropout(out).reshape(b, -1, self.d_head*self.n_head)
        #print(out.shape)
        out = self.out(out)
        return out, att

## Encoder

In [38]:
class Encoder(nn.Module):
    def __init__(self, d_model=16, vocab_size=20, seq_len=32, nb_layers=5, n_head=2, d_head=8):
        super().__init__()
        self.embedding = EmbeddingPosicional(d_model, vocab_size, seq_len)
        self.ff = nn.ModuleList([FeedForward(d_model) for _ in range(nb_layers)])
        self.atts = nn.ModuleList([SelfAttention(d_model, n_head, d_head) for _ in range(nb_layers)])
        self.norm_final = nn.LayerNorm(d_model)
    
    def forward(self, x):
        x = self.embedding(x)
        for ff_layer, att_layer in zip(self.ff, self.atts):
            att_out, _ = att_layer(x)
            x = x + att_out
            ff_out = ff_layer(x)
            x = x + ff_out
        x = self.norm_final(x)
        return x

In [39]:
encoder = Encoder()
x = torch.randint(0, 20, (2, 32))
out = encoder(x)
print(out.shape)

torch.Size([2, 32, 16])


## Tokenizador

In [ ]:
import re
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

class SimpleTokenizer:
    def __init__(self, text_corpus):
        tokens = word_tokenize(text_corpus.lower())
        
        self.vocab = sorted(list(set([t for t in tokens if t.isalnum()])))
        
        self.vocab_size = len(self.vocab) + 3 
        self.word2idx = {w: i+3 for i, w in enumerate(self.vocab)}
        self.idx2word = {i+3: w for i, w in enumerate(self.vocab)}
        
        self.PAD_TOKEN = 0
        self.MASK_TOKEN = 1
        self.UNK_TOKEN = 2
        self.word2idx['[PAD]'] = 0
        self.word2idx['[MASK]'] = 1
        self.word2idx['[UNK]'] = 2

    def encode(self, text, max_len):
        tokens = word_tokenize(text.lower())
        indices = [self.word2idx.get(t, self.UNK_TOKEN) for t in tokens if t.isalnum()]
        
        if len(indices) < max_len:
            indices += [self.PAD_TOKEN] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
        return torch.tensor(indices, dtype=torch.long)
    

# Clase para el Dataset

In [41]:
class BookDataset(Dataset):
    def __init__(self, pages, tokenizer, seq_len):
        self.pages = pages
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.pages)

    def __getitem__(self, idx):
        return self.tokenizer.encode(self.pages[idx], self.seq_len)

## Modelo Final

In [42]:
class EncoderForMaskedLM(nn.Module):
    def __init__(self, encoder, d_model, vocab_size):
        super().__init__()
        self.encoder = encoder
        # Capa extra para decodificar el embedding de vuelta a palabras
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x: (batch, seq_len)
        embeddings = self.encoder(x) # (batch, seq_len, d_model)
        prediction_scores = self.head(embeddings) # (batch, seq_len, vocab_size)
        return prediction_scores, embeddings

Función para entrenar al modelo

En esta función en vez de solo enmascarar los tokens, se puede aplicar también un cambio por otro token del vocabulario o mantenerse igual

In [ ]:
from tqdm import tqdm 
def train_model(pages, d_model=256, seq_len=20, epochs=100):
    full_text = " ".join(pages)
    tokenizer = SimpleTokenizer(full_text)
    dataset = BookDataset(pages, tokenizer, seq_len)
    
    # Drop_last=True ayuda si el último batch es muy pequeño e inestable
    loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)
    
    print(f"Vocab Size: {tokenizer.vocab_size}")

    base_encoder = Encoder(d_model=d_model, vocab_size=tokenizer.vocab_size, 
                           seq_len=seq_len, nb_layers=6, n_head=8, d_head=d_model//8)
    
    model = EncoderForMaskedLM(base_encoder, d_model, tokenizer.vocab_size)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    
    optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)

    model.train()
    
    for epoch in tqdm(range(epochs)):
        total_loss = 0
        for batch in loader:
            inputs = batch.to(device) 
            labels = batch.clone().to(device) 
                    
            # bert original
            for i in range(inputs.size(0)): 
                for j in range(inputs.size(1)):
                    token_id = inputs[i, j].item()
                    # Si token es PAD, no hacer nada
                    if token_id == tokenizer.PAD_TOKEN:
                        labels[i, j] = -100
                        continue
                    # Aplicar máscara con probabilidad 15%
                    prob = np.random.rand()
                    if prob < 0.15:
                        r = np.random.rand()
                        if r < 0.8: # 80% de las veces, reemplazar por [MASK]
                            inputs[i, j] = tokenizer.MASK_TOKEN
                        elif r < 0.9: # 10% de las veces, reemplazar por token aleatorio
                            random_token = np.random.randint(0, tokenizer.vocab_size)
                            inputs[i, j] = random_token
                        else: # 10% de las veces, dejar
                            pass
                    else: # No enmascarar este token si no se selecciona
                        labels[i, j] = -100
            
            outputs, _ = model(inputs) 
            
            loss = criterion(outputs.view(-1, tokenizer.vocab_size), labels.view(-1))
            
            optimizer.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_loss = total_loss / len(loader)
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    
    torch.save(model.state_dict(), "encoder_mlm_bert.pth")

    return model.encoder, tokenizer

Función para obtener los embeddings

In [44]:
import torch.nn.functional as F
def get_page_embeddings(encoder, tokenizer, pages, seq_len):
    encoder.eval()
    embeddings = []
    
    with torch.no_grad():
        for page in pages:
            # Codificar y mover a GPU
            input_tensor = tokenizer.encode(page, seq_len).unsqueeze(0).to(device)
            
            # Obtener salidas del encoder (1, seq_len, d_model)
            page_emb_seq = encoder(input_tensor)
            
            mask = (input_tensor != tokenizer.PAD_TOKEN).unsqueeze(-1) # (1, seq_len, 1)
            
            masked_embeddings = page_emb_seq * mask.float()
            
            sum_embeddings = masked_embeddings.sum(dim=1)
            
            token_count = mask.sum(dim=1).clamp(min=1e-9)
            
            page_emb_vector = sum_embeddings / token_count
            mean_embedding = F.normalize(page_emb_vector, p=2, dim=1)
            
            embeddings.append(mean_embedding.squeeze(0).cpu().numpy())            
    return embeddings

## Leer los pdfs

In [ ]:
import glob
import pypdf
import re

seq_len = 256
overlap = 10  # Solapamiento de palabras para mantener contexto entre chunks

def clean_page_text(text):
    """
    Limpia el texto crudo extraído del PDF antes de dividirlo.
    """
    if not text: return ""
    
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    text = re.sub(r'[^\w\s.,!?áéíóúÁÉÍÓÚñÑüÜ]', '', text) 

    tokens = word_tokenize(text.lower())
    
    clean_tokens = [t for t in tokens if t.isalnum()]
    
    return clean_tokens

# Cargar páginas
pdf_files = glob.glob("/home/alumno.upv.es/dnargar/tlh_trabajo/pdfs/*.pdf")

chunks_pdfs = {}

total_pages = 0

for pdf_file in pdf_files:
    try:
        reader = pypdf.PdfReader(pdf_file)
        file_chunks = []
        
        for page in tqdm(reader.pages):
            raw_text = page.extract_text()
            
            # Limpieza inicial
            clean_text = clean_page_text(raw_text)
            
            if not clean_text:
                continue
                
            words = clean_text
            
            # Si la página tiene muy poco contenido se salta
            if len(words) < 10:
                continue
                
            step = seq_len - overlap
            if step < 1: step = 1
            
            for i in range(0, len(words), step):
                # Extraer chunk
                chunk_words = words[i : i + seq_len]
                if len(chunk_words) >= 2 and chunk_words[-2] == "página" and chunk_words[-1].isdigit():
                    chunk_words = chunk_words[:-2]
                # Reconstruir string
                chunk = " ".join(chunk_words)
                
                if len(chunk_words) < 15: # Descartar restos muy pequeños al final
                    continue
                
                file_chunks.append(chunk)
                
        if file_chunks:
            chunks_pdfs[pdf_file] = file_chunks
            print(f"Procesado {pdf_file}: {len(file_chunks)} chunks.")
            
    except Exception as e:
        print(f"Error leyendo {pdf_file}: {e}")

# Verificación
all_chunks = []
for k, v in chunks_pdfs.items():
    all_chunks.extend(v)

print(f"\nTotal chunks limpios: {len(all_chunks)}")
if len(all_chunks) > 0:
    print("Ejemplo de chunk limpio:")
    print(f"'{all_chunks[5]}'")

100%|██████████| 544/544 [00:20<00:00, 26.50it/s]


Procesado /home/alumno.upv.es/dnargar/tlh_trabajo/pdfs/El Imperio Final.pdf: 1055 chunks.


100%|██████████| 607/607 [00:23<00:00, 25.95it/s]


Procesado /home/alumno.upv.es/dnargar/tlh_trabajo/pdfs/El Heroe de las Eras. (Ed. revisada).pdf: 1167 chunks.


100%|██████████| 639/639 [00:23<00:00, 26.95it/s]

Procesado /home/alumno.upv.es/dnargar/tlh_trabajo/pdfs/El Pozo de la Ascension. (Ed. revisada).pdf: 1216 chunks.

Total chunks limpios: 3438
Ejemplo de chunk limpio:
'son capaces de dominarlos todos ahora kelsier el superviviente el único que ha logrado huir de los pozos de hathsin ha encontrado a vin una pobre chica skaa con mucha suerte tal vez los dos unidos a la rebelión que los skaa intentan desde hace mil años puedan cambiar el mundo y la atroz dominación del lord legislador tras el sorprendente éxito mundial de elantris brandon sanderson ha logrado repetir la hazaña la trilogía nacidos de la bruma mistborn es una excepcional muestra de su nueva fantasía y el primer volumen el imperio final con una conclusión cerrada y grandiosa que permite su lectura como una novela única e independiente nos introduce gratamente en el mundo de los poderes alománticos de la bruma y la ceniza de la dominación y la rebelión con personajes inolvidables como el atormentado keisler en su papel de pigm

Funciones para obtener todos los chunks, la función para realizar la comparación entre embeddings y para obtener los chunks más importantes

In [46]:
def get_all_chunks(chunks_dict):
    all_chunks = []
    for chunks in chunks_dict.values():
        all_chunks.extend(chunks)
    return all_chunks

def cosine_similarity(v1, v2):
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0: return 0.0
    return np.dot(v1, v2) / (norm_v1 * norm_v2)

def get_relevant_chunks(query, encoder, tokenizer, chunks, seq_len, top_k=5):
    # La query también pasa por el proceso de embeddings (GPU -> CPU)
    query_emb = get_page_embeddings(encoder, tokenizer, [query], seq_len=seq_len)[0]
    
    similarities = []
    for i, chunk in enumerate(chunks):
        chunk_emb = get_page_embeddings(encoder, tokenizer, [chunk], seq_len=seq_len)[0]
        sim = cosine_similarity(query_emb, chunk_emb)
        similarities.append((sim, chunk))
    
    similarities.sort(reverse=True, key=lambda x: x[0])
    return similarities[:top_k]

## Entrenamos el modelo

In [15]:
d_model = 128

def get_all_chunks(chunks_dict):
    all_chunks = []
    for chunks in chunks_dict.values():
        all_chunks.extend(chunks)
    return all_chunks

all_chunks_pdfs = get_all_chunks(chunks_pdfs)


# Entrenar
encoder_entrenado, tokenizer = train_model(all_chunks_pdfs, d_model=d_model, seq_len=seq_len, epochs=50)

question = "la guerra contra los skaa"
print(f"\nPregunta: {question}")
print(tokenizer.encode(question, seq_len))

relevant_chunks = get_relevant_chunks(question, encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len=seq_len, top_k=3)

embedding_question = get_page_embeddings(encoder_entrenado, tokenizer, [question], seq_len=seq_len)[0]

for score, chunk in relevant_chunks:
    print(f"Similitud: {score:.4f}, Chunk: {chunk}")

Vocab Size: 27396



  0%|          | 0/50 [00:00<?, ?it/s]


  2%|▏         | 1/50 [00:21<17:37, 21.57s/it]


  4%|▍         | 2/50 [00:43<17:18, 21.63s/it]


  6%|▌         | 3/50 [01:04<16:50, 21.51s/it]

...


 98%|█████████▊| 49/50 [17:27<00:21, 21.30s/it]


100%|██████████| 50/50 [17:48<00:00, 21.31s/it]


100%|██████████| 50/50 [17:48<00:00, 21.37s/it]

Epoch 50, Loss: 5.9008

Pregunta: la guerra contra los skaa
tensor([15774, 13458,  6415, 16454, 24294,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
    

Similitud: 0.6707, Chunk: se volvió los ojos inyectados en sangre brillaban a la luz del incendio el frenesí los excitaba ante la perspectiva de un desafío elend tendría que asustarlos primero antes de poder tomar el control esta vez anhelaba hacerlo
Similitud: 0.6597, Chunk: que quería que estuviera cerca del refugio enséñamelo dijo vin saltando del edificio los skaa congregados seguían gritando cada grito más fuerte que el anterior toda la calle ardía de luz y las fluctuantes antorchas convertían la bruma en una llamarada brillante dockson la condujo calle abajo mientras el resto de la banda los seguía el almacén era una estructura grande y olvidada que se alzaba en la sección industrial de los skaa vin se acercó avivó peltre y rompió el cerrojo la puerta se abrió lentamente dockson alzó una linterna y su luz reveló el destello de montones de metal armas espadas hachas bastones y cascos brillaban a la luz un increíble alijo plateado todos se quedaron mirando asombrados
Similitud: 0.6

## Obtenemos los embeddings del modelo y los guardamos

In [ ]:
import torch
import numpy as np
from sklearn.decomposition import PCA

def guardar_comparativa_embeddings(encoder, tokenizer, seq_len, file_antes="antes.txt", file_despues="despues.txt"):
    encoder.eval()
    vocab = tokenizer.word2idx
    palabras = []
    embs_antes = []
    embs_despues = []

    print(f"Procesando {len(vocab)} palabras...")

    with torch.no_grad():
        for palabra, idx in vocab.items():
            if palabra == "[PAD]": continue
            
            input_tensor = tokenizer.encode(palabra, seq_len).unsqueeze(0).to(device)
            
            antes = encoder.embedding(input_tensor) # (1, seq_len, d_model)
            
            despues = encoder(input_tensor) # (1, seq_len, d_model)
            
            palabras.append(palabra)
            embs_antes.append(antes[0, 0, :].cpu().numpy())
            embs_despues.append(despues[0, 0, :].cpu().numpy())

    # Reducción a 3 dimensiones mediate PCA
    pca_antes = PCA(n_components=3)
    pca_despues = PCA(n_components=3)
    
    res_antes = pca_antes.fit_transform(np.array(embs_antes))
    res_despues = pca_despues.fit_transform(np.array(embs_despues))

    # Guardamos los datos en ficheros
    print(f"Guardando {file_antes}...")
    with open(file_antes, "w", encoding="utf-8") as f:
        for i, palabra in enumerate(palabras):
            vec = res_antes[i]
            f.write(f"{palabra} {vec[0]:.6f} {vec[1]:.6f} {vec[2]:.6f}\n")

    print(f"Guardando {file_despues}...")
    with open(file_despues, "w", encoding="utf-8") as f:
        for i, palabra in enumerate(palabras):
            vec = res_despues[i]
            f.write(f"{palabra} {vec[0]:.6f} {vec[1]:.6f} {vec[2]:.6f}\n")

    print("Proceso finalizado con éxito.")


guardar_comparativa_embeddings(encoder_entrenado, tokenizer, seq_len)

Procesando 27396 palabras...


Guardando antes.txt...
Guardando despues.txt...
Proceso finalizado con éxito.


# Pruebas top-k (K=1 K=3, K=5)

In [ ]:
import random

def evaluar_modelo(encoder, tokenizer, paginas, seq_len, k=5, num_muestras=50, ratio_query=0.5):
    """
    ratio_query: 0.5 significa que usamos la mitad del texto como consulta.
                 0.1 significa que usamos el 10% del texto.
    """
    print(f"--- Evaluación (Ratio: {ratio_query}, Top-K: {k}) ---")
    
    lista_numpy = get_page_embeddings(encoder, tokenizer, paginas, seq_len)
    db_embeddings = torch.tensor(np.array(lista_numpy))
    
    aciertos = 0
    indices_prueba = list(range(len(paginas)))
    random.shuffle(indices_prueba)
    indices_prueba = indices_prueba[:num_muestras]
    
    for idx_real in indices_prueba:
        texto_chunk = paginas[idx_real]
        palabras = texto_chunk.split()
        
        if len(palabras) < 10: continue

        # Calculamos longitud basada en el porcentaje
        target_len = int(len(palabras) * ratio_query)
        
        # Limites de la longitud de la query
        len_query = max(target_len, 5)
        len_query = min(len_query, seq_len - 2)
        
        # Si el chunk es mas corto que seq_len, limitamos
        limit_start = len(palabras) - len_query
        if limit_start <= 0: 
            start_idx = 0
            len_query = len(palabras) # Usar todo si es muy corto
        else:
            start_idx = random.randint(0, limit_start)
            
        palabras_query = palabras[start_idx : start_idx + len_query]
        query_text = " ".join(palabras_query)
        
        vec_query_np = get_page_embeddings(encoder, tokenizer, [query_text], seq_len)[0]
        vec_query = torch.tensor(vec_query_np).unsqueeze(0)
        
        similitudes = F.cosine_similarity(vec_query, db_embeddings)
        vals, top_indices = torch.topk(similitudes, k=k)
        
        if idx_real in top_indices.tolist():
            aciertos += 1

    accuracy = (aciertos / len(indices_prueba)) * 100
    print(f"Ratio {ratio_query} -> Precisión: {accuracy:.2f}%")
    return accuracy

In [ ]:
import os

if os.path.exists("encoder_mlm_bert.pth"):
    print("Cargando modelo entrenado desde disco...")
    d_model = 128
    seq_len = 256
    all_chunks_pdfs = get_all_chunks(chunks_pdfs)
    full_text = " ".join(all_chunks_pdfs)
    tokenizer = SimpleTokenizer(full_text)
    base_encoder = Encoder(d_model=d_model, vocab_size=tokenizer.vocab_size, 
                           seq_len=seq_len, nb_layers=6, n_head=8, d_head=d_model//8)
    encoder_entrenado = EncoderForMaskedLM(base_encoder, d_model, tokenizer.vocab_size)
    encoder_entrenado.load_state_dict(torch.load("encoder_mlm_bert.pth", map_location=device))
    encoder_entrenado.to(device)
    encoder_entrenado.eval()
    encoder_entrenado = encoder_entrenado.encoder

    print("EXPERIMENTOS K=1")

    print("EXPERIMENTO 1: Query Larga (50% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.5)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.3)
                        for _ in range(3)]) / 3

    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.1)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("EXPERIMENTOS K=3")

    print("EXPERIMENTO 1: Query Larga (50% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.5)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.3)
                        for _ in range(3)]) / 3

    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.1)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")


    print("EXPERIMENTOS K=5")
    print("EXPERIMENTO 1: Query Larga (50% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.5)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.3)
                        for _ in range(3)]) / 3

    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.1)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

Cargando modelo entrenado desde disco...
EXPERIMENTOS K=1
EXPERIMENTO 1: Query Larga (50% del texto original)
--- Evaluación (Ratio: 0.5, Top-K: 1) ---
Ratio 0.5 -> Precisión: 96.00%
--- Evaluación (Ratio: 0.5, Top-K: 1) ---
Ratio 0.5 -> Precisión: 96.00%
--- Evaluación (Ratio: 0.5, Top-K: 1) ---
Ratio 0.5 -> Precisión: 92.00%
Precisión promedio (3 ejecuciones): 94.67%

EXPERIMENTO 2: Query Media (30% del texto original)
--- Evaluación (Ratio: 0.3, Top-K: 1) ---
Ratio 0.3 -> Precisión: 64.00%
--- Evaluación (Ratio: 0.3, Top-K: 1) ---
Ratio 0.3 -> Precisión: 46.00%
--- Evaluación (Ratio: 0.3, Top-K: 1) ---
Ratio 0.3 -> Precisión: 50.00%
Precisión promedio (3 ejecuciones): 53.33%

EXPERIMENTO 3: Query Corta (10% del texto original - Difícil)
--- Evaluación (Ratio: 0.1, Top-K: 1) ---
Ratio 0.1 -> Precisión: 2.00%
--- Evaluación (Ratio: 0.1, Top-K: 1) ---
Ratio 0.1 -> Precisión: 4.00%
--- Evaluación (Ratio: 0.1, Top-K: 1) ---
Ratio 0.1 -> Precisión: 10.00%
Precisión promedio (3 ejecuciones